In [0]:
-- ============================================================
-- NOTEBOOK: 00_metadata_config.sql
-- PURPOSE : Central metadata registry and environment setup.
--
-- CATALOG STRUCTURE:
--   yt_dev.bronze   | yt_dev.silver   | yt_dev.gold
--   yt_uat.bronze   | yt_uat.silver   | yt_uat.gold
--   yt_prod.bronze  | yt_prod.silver  | yt_prod.gold
--
-- VOLUME  : /Volumes/{catalog}/bronze/raw/
-- AUTHOR  : The Data Master | Naval Yemul
-- ============================================================

-- ─────────────────────────────────────────────────────────────
-- STEP 1 : WIDGET  (env drives everything — one param only)
-- ─────────────────────────────────────────────────────────────
-- In Python cell at top of every notebook:
--
--   dbutils.widgets.dropdown("env", "yt_dev", ["yt_dev", "yt_uat", "yt_prod"])
--
--   env         = dbutils.widgets.get("env")   # yt_dev | yt_uat | yt_prod
--   catalog     = env                           # catalog name IS the env
--   volume_path = f"/Volumes/{catalog}/bronze/raw"
--
--   spark.conf.set("pipeline.catalog",     catalog)
--   spark.conf.set("pipeline.volume_path", volume_path)
--
-- All SQL cells use ${catalog} — Databricks resolves from spark.conf.
-- Lakeflow Job passes env as a task parameter; no widget needed.


-- ─────────────────────────────────────────────────────────────
-- STEP 2 : CREATE SCHEMAS IN ALL CATALOGS (one-time setup)
-- ─────────────────────────────────────────────────────────────
-- Run this once per workspace. Assumes catalogs yt_dev/yt_uat/yt_prod
-- already exist in Unity Catalog.

-- yt_dev
CREATE SCHEMA IF NOT EXISTS yt_dev.bronze  COMMENT 'Bronze layer — yt_dev catalog';
CREATE SCHEMA IF NOT EXISTS yt_dev.silver  COMMENT 'Silver layer — yt_dev catalog';
CREATE SCHEMA IF NOT EXISTS yt_dev.gold    COMMENT 'Gold layer   — yt_dev catalog';

-- yt_uat
CREATE SCHEMA IF NOT EXISTS yt_uat.bronze  COMMENT 'Bronze layer — yt_uat catalog';
CREATE SCHEMA IF NOT EXISTS yt_uat.silver  COMMENT 'Silver layer — yt_uat catalog';
CREATE SCHEMA IF NOT EXISTS yt_uat.gold    COMMENT 'Gold layer   — yt_uat catalog';

-- yt_prod
CREATE SCHEMA IF NOT EXISTS yt_prod.bronze COMMENT 'Bronze layer — yt_prod catalog';
CREATE SCHEMA IF NOT EXISTS yt_prod.silver COMMENT 'Silver layer — yt_prod catalog';
CREATE SCHEMA IF NOT EXISTS yt_prod.gold   COMMENT 'Gold layer   — yt_prod catalog';


-- ─────────────────────────────────────────────────────────────
-- STEP 3 : METADATA TABLE  (lives in yt_prod.bronze — single source)
-- ─────────────────────────────────────────────────────────────
-- All environments share this registry; env is passed at runtime.
-- Add a new row here to onboard a new source table —
-- no notebook code changes required.

CREATE TABLE IF NOT EXISTS yt_prod.bronze.ingestion_metadata (
  table_name          STRING   COMMENT 'Target Delta table name in Bronze',
  source_file_name    STRING   COMMENT 'CSV filename inside /Volumes/{catalog}/bronze/raw/',
  primary_key_col     STRING   COMMENT 'Business key used for dedup in Silver',
  partition_col       STRING   COMMENT 'Partition column (NULL = unpartitioned)',
  load_type           STRING   COMMENT 'full | incremental',
  silver_dedup_order  STRING   COMMENT 'ORDER BY clause used for Silver ROW_NUMBER()',
  is_active           BOOLEAN  COMMENT 'Set false to skip without deleting the row',
  created_by          STRING,
  created_at          TIMESTAMP
)
USING DELTA
COMMENT 'Central ingestion registry — shared across yt_dev / yt_uat / yt_prod.';

-- Seed rows (idempotent)
INSERT INTO yt_prod.bronze.ingestion_metadata VALUES
  ('raw_transactions', 'raw_transactions.csv', 'transaction_id', 'transaction_date', 'full', 'transaction_date DESC, _ingested_at DESC', true, 'naval.yemul', current_timestamp()),
  ('raw_customers',    'raw_customers.csv',    'customer_id',    NULL,               'full', 'created_date DESC, _ingested_at DESC',    true, 'naval.yemul', current_timestamp()),
  ('raw_accounts',     'raw_accounts.csv',     'account_id',     NULL,               'full', 'open_date DESC, _ingested_at DESC',        true, 'naval.yemul', current_timestamp()),
  ('raw_loans',        'raw_loans.csv',        'loan_id',        NULL,               'full', 'disbursement_date DESC, _ingested_at DESC',true, 'naval.yemul', current_timestamp())
;

SELECT * FROM yt_prod.bronze.ingestion_metadata ORDER BY table_name;